In [21]:
import sys
sys.path.append("..")
import torch
from components.model import GPTModel

In [22]:
GPT_CONFIG_124M = {
"vocab_size": 50257,
"context_length": 256,
"emb_dim": 768,
"n_heads": 12,
"n_layers": 12,
"drop_rate": 0.1,
"qkv_bias": False
}

torch.manual_seed(123)
model= GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [23]:
import tiktoken
from components.model import generate_text_simple

def text_to_token_ids(text, tokenizer):
    encoded= tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor =torch.tensor(encoded).unsqueeze(0)  #ads batch dimension
    return encoded_tensor
def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist()) #removes batch dimension

start_context ="Attention is all you need"
tokenizer= tiktoken.get_encoding("gpt2")
token_ids= generate_text_simple(model=model, idx=text_to_token_ids(start_context, tokenizer), max_new_tokens=10, context_size=GPT_CONFIG_124M["context_length"])

print("Output text: \n", token_ids_to_text(token_ids, tokenizer))

Output text: 
 Attention is all you need lettuce Imperium96 VIDEOS Royale sanct sellers Supportedettingatural


In [24]:
# text gen loss
inputs = torch.tensor([[16833, 3626, 6100], # ["every effort moves",
[40, 1107, 588]]) # "I really like"]
targets = torch.tensor([[3626, 6100, 345 ], # [" effort moves you",
[1107, 588, 11311]]) # " really like chocolate"]
with torch.no_grad():
    logits= model(inputs)
probas= torch.softmax(logits, dim=-1)
print(probas.shape)     

torch.Size([2, 3, 50257])


In [25]:
token_ids= torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs: ", token_ids)

Token IDs:  tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


In [26]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1:"f" {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1:  Armed heNetflix


In [27]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)
text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

Text 1: tensor([7.4540e-05, 3.1061e-05, 1.1563e-05])
Text 2: tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])
tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])


In [28]:
print("Logits shape:", logits.shape)
print("Targets shape:", targets.shape)

Logits shape: torch.Size([2, 3, 50257])
Targets shape: torch.Size([2, 3])


In [29]:
logits_flat =logits.flatten(0, 1)
targets_flat = targets.flatten()
print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])


In [30]:
loss =torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(10.7940)


In [32]:
file_path= "the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
    text_data=file.read()

In [33]:
total_chars =len(text_data)
total_tokens =len(tokenizer.encode(text_data))
print("Characters: ", total_chars)
print("Tokens: ", total_tokens)

Characters:  20479
Tokens:  5145


In [34]:
# training and validation dataset
train_ratio =0.9 #90% of data fr trraining and rest for validation
split_idx =int(train_ratio*len(text_data))
train_data =text_data[:split_idx]
val_data =text_data[split_idx:]

In [38]:
from utils.dataloader import create_dataloader_v1
torch.manual_seed(123)
 
train_loader= create_dataloader_v1(train_data, batch_size=2, max_length=GPT_CONFIG_124M["context_length"], stride=GPT_CONFIG_124M["context_length"], drop_last=True, shuffle=True, num_workers=0)
val_loader= create_dataloader_v1(train_data, batch_size=2, max_length=GPT_CONFIG_124M["context_length"], stride=GPT_CONFIG_124M["context_length"], drop_last=False, shuffle=False, num_workers=0)


In [39]:
print("Train Loader: ")
for x, y in train_loader:
    print(x.shape, y.shape)
print("Validation Loader: ")
for x, y in val_loader:
    print(x.shape, y.shape)

Train Loader: 
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
Validation Loader: 
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])


In [40]:
# calculatin cross entropy loss of given batch
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch= input_batch.to(device)
    target_batch =target_batch.to(device)
    logits =model(input_batch)
    loss =torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

In [ ]:
# computing training and validation loss
def calc_loss_loader(data_loader,model, device, num_batches=None):
    total_loss =0
    if len(data_loader)== 0:
        return float("nan")
    elif num_batches is None:
        num_batches =len(data_loader)
    else:
        num_batches =min(num_batches, len(data_loader))

    # reduces no of batches to match total no of batches in data loader if num_batches exceeds no of batches in data loader
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i <num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss +=loss.item()
        else:
            break
    return total_loss/ num_batches

In [42]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    train_loss= calc_loss_loader(train_loader, model, device)
    val_loss =calc_loss_loader(val_loader, model, device)

print("Training loss: ", train_loss)
print("Validation loss: ", val_loss)    

Training loss:  10.987583584255642
Validation loss:  10.987583690219456
